# OMNI-917 · Measures that fit forecasting, and which model the product serves (Colab GPU)

Measuring only, on **validation**: no training. Both models' three seeds (`best.pt` already on Drive), both
domains, D-64's five measures (`scripts/pfm_measures.py`, written before any of these numbers was computed):

1. **Within-window AUC:** inside each event window, samples close to the event (UCF-Crime τ in [0, 10) s, CCD
   [0, 1) s) against earlier ones of the same window (UCF-Crime [20, 30) s, CCD [2.5, 5) s)
2. **Flat-line check:** the same inside normal windows (time before the window's last sample); about 0.5 is right
3. **Per-window AUC:** each window's highest score, event windows against normal ones
4. **AP and mTTA** over every warning line, and the detection rate at `θ_d` (the 10% rule)
5. **The pooled AUC** as before

**D-64's rule:** the event-only model (`pfm-joint-v3-ev-s0-e118`) replaces v2 (`pfm-joint-v2-mf-s2-e55`) only if
its UCF-Crime within-window AUC beats v2's by **more than the larger seed spread** of that measure **and** its
flat-line check is **≤ 0.60**. Otherwise v2 stays. No seed is re-picked, `θ_d` unchanged.

1. **Setup:** clone, Drive
2. **The vectors:** as OMNI-905
3. **The measures:** 2 models × 3 seeds × 2 domains
4. **The rule and the record:** `model_runs.json` (only if the served model changes), `results/omni917_measures.json`

Then held-out once, only for the served model: `notebooks/pfm_heldout_gpu.ipynb` with its run id.
Runtime → Change runtime type → **GPU** (a T4 is enough: evaluation only).

In [ ]:
# 1 · Setup
BRANCH = "slice-09-10-forecast"
REPO = "https://github.com/OmniSight-team/omnisight.git"

import os, subprocess, sys
if not os.path.isdir("/content/omni-sight"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, "/content/omni-sight"], check=True)
else:
    subprocess.run(["git", "-C", "/content/omni-sight", "pull", "-q"], check=True)
COMMIT = subprocess.run(["git", "-C", "/content/omni-sight", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print(COMMIT)
sys.path.insert(0, "/content/omni-sight/scripts")

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import torch
assert torch.cuda.is_available(), "no GPU: Runtime -> Change runtime type -> GPU"
DEVICE, GPU = "cuda", torch.cuda.get_device_name(0)
print(torch.__version__, GPU)

In [ ]:
# 2 · Both domains' InternVideo2 vectors, copied from Drive once, and the samples
import hashlib, json, os, shutil, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
import pfm_runs, pfm_samples, train_pfm
from pfm_model import Config

REPO_DIR = Path("/content/omni-sight")
DRIVE = Path("/content/drive/MyDrive/omnisight")
LOCAL = Path("/content/pfm")
DOMAINS = ("ccd", "ucf_crime")
SAMPLE_FILES = {d: REPO_DIR / "datasets" / "samples" / f"{d}.csv" for d in DOMAINS}
SAMPLES = {d: pfm_samples.read_samples(SAMPLE_FILES[d]) for d in DOMAINS}

def copy_folder(domain):
    keys = sorted({s["clip_key"] for s in SAMPLES[domain]})
    source, target = DRIVE / "embeddings" / "pfm" / domain / "internvideo2", LOCAL / domain
    def one(key):
        dst = target / f"{key}.npz"
        if not dst.is_file():  # copied under a temporary name, renamed when complete: a cut copy leaves no broken file
            dst.parent.mkdir(parents=True, exist_ok=True)
            part = dst.with_name(dst.name + ".part")
            shutil.copyfile(source / f"{key}.npz", part)
            os.replace(part, dst)
    with ThreadPoolExecutor(16) as pool:
        list(pool.map(one, keys))
    return target

STORES, TRAIN, VAL = {}, {}, {}
for d in DOMAINS:
    start = time.time()
    STORES[d] = pfm_samples.Store(SAMPLES[d], copy_folder(d))
    TRAIN[d] = train_pfm.Data.split(STORES[d], SAMPLES[d], "train", d)
    VAL[d] = train_pfm.Data.split(STORES[d], SAMPLES[d], "validation", d)
    print(f"{d}: {len(STORES[d].vectors):,} vectors, train {len(TRAIN[d]):,} samples, validation {len(VAL[d]):,} "
          f"({time.time() - start:.0f} s)")

In [ ]:
# 3 · D-64's measures on validation: both models, every seed, both domains
import pfm_measures
MODELS = {"v2": ("joint-minus-first-p30", "pfm-joint-v2-mf-s2-e55", 2),  # folder, served run, its seed
          "event": ("joint-event", "pfm-joint-v3-ev-s0-e118", 0)}
SEEDS = (0, 1, 2)
INDEX = json.loads((DRIVE / pfm_runs.INDEX).read_text(encoding="utf-8"))
RECORDS = {r["run_id"]: r for r in INDEX["runs"]}
TABLE = {}
for name, (folder, run_id, served) in MODELS.items():
    for seed in SEEDS:
        path = DRIVE / "models" / "pfm" / folder / f"seed{seed}" / "best.pt"
        if seed == served:  # the served run's model.pt is a copy of this best.pt
            assert pfm_runs.sha256(path) == RECORDS[run_id]["checkpoint_sha256"], f"{path} is not {run_id}"
        model, state = train_pfm.load(path, DEVICE)
        row = {"epoch": state["epoch"], "labels": model.config.labels}
        for d in DOMAINS:
            data = VAL[d].for_labels(model.config.labels)
            p, dt = train_pfm.predict(model, data, DEVICE)
            e = train_pfm.evaluate(data, p, dt)
            row[d] = {**pfm_measures.measures(data, p), "theta": e["theta"],
                      "detection_rate": e["warned_windows"] / e["positive_windows"],
                      "false_alarm_windows": e["false_alarm_windows"]}
        TABLE[f"{name} s{seed}"] = row

def f(v, spec=".4f"):
    return "-" if v is None else format(v, spec)

for d in DOMAINS:
    print(f"\n{d} (validation)")
    print("  model     epoch  within  (windows/skipped)  flat-line  per-window  AP      mTTA     detection  pooled")
    for key, row in TABLE.items():
        m = row[d]
        print(f"  {key:9s} {row['epoch']:5d}  {f(m['within_window_auc'])}  ({m['within_window_windows']}/"
              f"{m['within_window_skipped']})  {f(m['flat_line']):>15s}  {f(m['per_window_auc']):>10s}  {f(m['ap'])}  "
              f"{f(m['mtta_s'], '.2f'):>5s} s  {m['detection_rate']:.3f}      {f(m['pooled_auc'])}")

In [ ]:
# 4 · D-64's rule on UCF-Crime's validation, and the record
WITHIN = {n: [TABLE[f"{n} s{s}"]["ucf_crime"]["within_window_auc"] for s in SEEDS] for n in MODELS}
SPREAD = {n: max(v) - min(v) for n, v in WITHIN.items()}
MARGIN = max(SPREAD.values())
SERVED = {n: TABLE[f"{n} s{MODELS[n][2]}"]["ucf_crime"] for n in MODELS}
GAIN = SERVED["event"]["within_window_auc"] - SERVED["v2"]["within_window_auc"]
FLAT = SERVED["event"]["flat_line"]
WINS = GAIN > MARGIN and FLAT <= 0.60
CHOSEN = MODELS["event" if WINS else "v2"][1]
print(f"UCF-Crime within-window AUC: event-only {SERVED['event']['within_window_auc']:.4f} vs v2 "
      f"{SERVED['v2']['within_window_auc']:.4f} ({GAIN:+.4f}; needs more than +{MARGIN:.4f}; spreads "
      f"v2 {SPREAD['v2']:.4f}, event-only {SPREAD['event']:.4f}); flat-line {FLAT:.4f} (≤ 0.60; v2 "
      f"{SERVED['v2']['flat_line']:.4f}) -> {'the event-only model is served' if WINS else 'v2 stays served'}")
if INDEX["chosen"] != CHOSEN:
    pfm_runs.register(DRIVE, RECORDS[CHOSEN], chosen=True)
    print("model_runs.json: chosen ->", CHOSEN)
OUTDIR = DRIVE / "results"
OUTDIR.mkdir(parents=True, exist_ok=True)
(OUTDIR / "omni917_measures.json").write_text(json.dumps({
    "commit": COMMIT, "gpu": GPU, "torch": torch.__version__, "split": "validation", "ranges": pfm_measures.RANGES,
    "table": TABLE, "within_window_ucf_crime": WITHIN, "spread": SPREAD, "margin": MARGIN, "gain": GAIN,
    "flat_line_event": FLAT, "d64_rule_met": WINS, "chosen": CHOSEN,
    "samples_sha256": {d: hashlib.sha256(SAMPLE_FILES[d].read_bytes()).hexdigest() for d in DOMAINS},
}, indent=1, default=float))
print(OUTDIR / "omni917_measures.json")